# Neev ML Module — Working Demo

**Purpose:** predict 30-day failure risk and show the prediction handoff used by Arnav's optimizer.

Models included: CatBoost failure-risk classifier + HistGradientBoosting 30-day degradation model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import joblib

BASE = Path.cwd()
DATA = BASE / 'neev_ml_dataset.csv'
MODEL_DIR = BASE / 'models'
OUTPUT_DIR = BASE / 'output'

df = pd.read_csv(DATA)
print('Dataset:', df.shape)
display(df.head())

## 1. Load Neev's trained models

In [ ]:
risk_model = CatBoostClassifier()
risk_model.load_model(MODEL_DIR / 'neev_failure_risk_model.cbm')
print('Risk model:', type(risk_model).__name__)

# The degradation regressor was pickled by an older scikit-learn. Loading it under
# a much newer version raises ModuleNotFoundError, so this is reported rather than
# allowed to abort the demo -- the classifier below is the model the optimizer
# actually depends on, and the regressor's forecasts are already materialised in
# output/neev_predictions_for_optimizer.csv.
deg_model = None
try:
    deg_bundle = joblib.load(MODEL_DIR / 'neev_degradation_30d_model.joblib')
    deg_model = deg_bundle['model'] if isinstance(deg_bundle, dict) else deg_bundle
    print('Degradation model:', type(deg_model).__name__)
except Exception as exc:
    print(f'Degradation model not loadable in this environment: {type(exc).__name__}: {exc}')
    print('Its forecasts are available in output/neev_predictions_for_optimizer.csv.')

## 2. Run a real prediction on one asset

In [ ]:
import json

# The model's feature schema is authoritative and lives in model_metadata.json.
# Deriving it here by exclusion produced 26 columns against a 29-feature model,
# because obs_month / obs_dayofweek / obs_hour are engineered from
# observation_date at training time and are not columns in the raw dataset.
meta = json.loads((OUTPUT_DIR / 'model_metadata.json').read_text())
features = meta['features']

sample = df.iloc[[0]].copy()

# Recreate the three engineered datetime features.
obs = pd.to_datetime(sample['observation_date'])
sample['obs_month'] = obs.dt.month
sample['obs_dayofweek'] = obs.dt.dayofweek
sample['obs_hour'] = obs.dt.hour

missing = [c for c in features if c not in sample.columns]
assert not missing, f'Dataset is missing model features: {missing}'
print(f'Model expects {len(features)} features; prepared {len(features)}.')

p = float(risk_model.predict_proba(sample[features])[:, 1][0])
risk_score = p * 100
risk_level = ('LOW' if risk_score <= 30 else 'MODERATE' if risk_score <= 60 else
              'HIGH' if risk_score <= 80 else 'CRITICAL')

print('Asset ID  :', sample['asset_id'].iloc[0])
print('Risk Score:', round(risk_score, 2))
print('Risk Level:', risk_level)

## 3. View Neev → Arnav handoff

In [ ]:
handoff = pd.read_csv(OUTPUT_DIR / 'neev_predictions_for_optimizer.csv')
print('Rows in optimizer handoff:', len(handoff))
display(handoff.head(10))

### Pipeline

`neev_ml_dataset.csv` → **Neev ML models** → `neev_predictions_for_optimizer.csv` → **Arnav OR-Tools optimizer**

The report, feature-importance file, and model files document/support Neev's work; the optimizer consumes the prediction CSV.